In [1]:
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import os
import urllib.request # Built-in Python library for network requests

# 1. Imports strictly from the new Tasks API (as per documentation)
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles

from tqdm import tqdm  # For progress bars in loops

# Allow matplotlib to display inline
%matplotlib inline

class PoseDetector:
    def __init__(self, model_asset_path='pose_landmarker_heavy.task'):
        # Pure Python fallback to download the model if missing
        if not os.path.exists(model_asset_path):
            print(f"Model not found. Downloading to {model_asset_path}...")
            url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task"
            urllib.request.urlretrieve(url, model_asset_path)
            print("✅ Download complete!")

        # 2. Create the PoseLandmarker object for VIDEO
        base_options = python.BaseOptions(model_asset_path=model_asset_path)
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.detector = vision.PoseLandmarker.create_from_options(options)

    def find_pose(self, img, timestamp_ms, draw=True):
        """Processes the image and draws landmarks using the new API utilities."""
        # Convert OpenCV BGR image to RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Load the input image into MediaPipe's Image object
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

        # Detect pose landmarks from the video frame
        self.results = self.detector.detect_for_video(mp_image, int(timestamp_ms))

        # Process the detection result and visualize it
        if self.results.pose_landmarks and draw:
            annotated_rgb = self._draw_landmarks_on_image(img_rgb, self.results)
            # Convert back to BGR so OpenCV can write it to video correctly
            return cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
            
        return img

    def _draw_landmarks_on_image(self, rgb_image, detection_result):
        """Visualization utility exactly as provided in the Colab documentation."""
        pose_landmarks_list = detection_result.pose_landmarks
        annotated_image = np.copy(rgb_image)

        pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
        pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

        for pose_landmarks in pose_landmarks_list:
            drawing_utils.draw_landmarks(
                image=annotated_image,
                landmark_list=pose_landmarks,
                connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
                landmark_drawing_spec=pose_landmark_style,
                connection_drawing_spec=pose_connection_style)

        return annotated_image

In [2]:
def inference_on_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print(f"❌ Error: Could not open video at {input_path}")
    else:
        detector = PoseDetector(model_asset_path='pose_landmarker_heavy.task')

        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

        progress_bar = tqdm(total=total_frames, desc="Analyzing Squat Form", unit="frames")

        while cap.isOpened():
            success, img = cap.read()
            if not success:
                break 
                
            # Get the actual video timestamp in milliseconds
            timestamp_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            
            # The API requires strictly increasing timestamps. 
            # If the video restarts or loops, this prevents a crash.
            if timestamp_ms < 0: 
                timestamp_ms = 0

            # Process the frame
            img = detector.find_pose(img, timestamp_ms, draw=True)
            
            out.write(img)
            progress_bar.update(1)

        progress_bar.close()
        cap.release()
        out.release()
        print(f"✅ Done! Processed video saved to: {output_path}")

In [3]:
# UPDATE THIS PATH to your test video
input_path = '/home/rahul/Projects/squat-form-analyzer/sample_vids/youTube_video.mp4' 
output_path = '/home/rahul/Projects/squat-form-analyzer/inferred_vid/youTube_video.mp4'

inference_on_video(input_path, output_path)

Model not found. Downloading to pose_landmarker_heavy.task...
✅ Download complete!


I0000 00:00:1775727281.957816  123956 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775727281.979890  123969 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.25.10.1), renderer: Mesa Intel(R) UHD Graphics 620 (KBL GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775727282.149858  123962 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775727282.297783  123957 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Analyzing Squat Form: 100%|██████████| 514/514 [01:47<00:00,  4.79frames/s]


✅ Done! Processed video saved to: /home/rahul/Projects/squat-form-analyzer/inferred_vid/youTube_video.mp4


In [5]:
# UPDATE THIS PATH to your test video
input_path = '/home/rahul/Projects/squat-form-analyzer/sample_vids/squat_demo.mp4' 
output_path = '/home/rahul/Projects/squat-form-analyzer/inferred_vid/squat_demo.mp4'

inference_on_video(input_path, output_path)

I0000 00:00:1775728622.008245  142126 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775728622.013279  142137 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.25.10.1), renderer: Mesa Intel(R) UHD Graphics 620 (KBL GT2)
W0000 00:00:1775728622.214611  142128 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775728622.388222  142130 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Analyzing Squat Form: 100%|██████████| 514/514 [01:54<00:00,  4.49frames/s]


✅ Done! Processed video saved to: /home/rahul/Projects/squat-form-analyzer/inferred_vid/squat_demo.mp4
